In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

| Column         | Type    | Description                                              | Unique Values / Range                     |
|----------------|---------|------------------------------------------------------------|--------------------------------------------|
| `user_id`      | int64   | Unique identifier for each user                            | 290,584 unique (out of 294,478 rows → some duplicates) |
| `timestamp`    | string  | Date and time the user visited the page                    | 2017-01-02 to 2017-01-24                  |
| `group`        | string  | Experiment arm the user was assigned to                    | `control`, `treatment`                    |
| `landing_page` | string  | Page version the user actually saw                         | `old_page`, `new_page`                    |
| `converted`    | int     | Whether the user converted (1) or not (0)                  | `0`, `1` (binary)                         |

In [40]:
df = pd.read_csv('ab_data.csv')
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [41]:
df.shape

(294478, 5)

In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294478 non-null  int64 
 1   timestamp     294478 non-null  object
 2   group         294478 non-null  object
 3   landing_page  294478 non-null  object
 4   converted     294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


In [43]:
#Data Type Correction
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   user_id       294478 non-null  int64         
 1   timestamp     294478 non-null  datetime64[ns]
 2   group         294478 non-null  object        
 3   landing_page  294478 non-null  object        
 4   converted     294478 non-null  int64         
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 11.2+ MB


In [45]:
df['group'].value_counts()

group
treatment    147276
control      147202
Name: count, dtype: int64

In [46]:
df['landing_page'].value_counts()

landing_page
old_page    147239
new_page    147239
Name: count, dtype: int64

In [47]:
df['user_id'].nunique()

290584

In [48]:
pd.crosstab(df['group'], df['landing_page'])

landing_page,new_page,old_page
group,,
control,1928,145274
treatment,145311,1965


In [49]:
match_mask = ((df['group'] == 'control') & (df['landing_page'] == 'old_page')) | \
             ((df['group'] == 'treatment') & (df['landing_page'] == 'new_page'))

df = df[match_mask].copy()

In [50]:
pd.crosstab(df['group'], df['landing_page'])

landing_page,new_page,old_page
group,,
control,0,145274
treatment,145311,0


In [54]:
print('Duplicate rows:', df.duplicated(subset='user_id').sum())

Duplicate rows: 1


In [55]:
df = df.drop_duplicates(subset='user_id', keep='first')

In [56]:
print('Shape after removing duplicates:', df.shape)

Shape after removing duplicates: (290584, 5)


# A/B Testing

In [57]:
control = df[df['group'] == 'control']['converted']
treatment = df[df['group'] == 'treatment']['converted']

In [58]:
n_control = len(control)
n_treatment = len(treatment)

print(n_control)
print(n_treatment)

145274
145310


In [59]:
conv_control = control.sum()
conv_treatment = treatment.sum()

print(conv_control)
print(conv_treatment)

17489
17264


In [64]:
p_control = conv_control / n_control*100
p_treatment = conv_treatment / n_treatment*100

print(p_control)
print(p_treatment)

12.03863045004612
11.880806551510565


## A/B Test Hypothesis — Conversion Rate

**Metric:** Conversion rate = (number of users who converted) / (total users), per group

**Hypothesis:** we make new page and may be it will make more conversions

**Null Hypothesis (H0):**
There is no difference in conversion rate between the new page and the old page.
> H0: p_treatment = p_control

**Alternative Hypothesis (H1):**
There is a difference in conversion rate between the new page and the old page.
> H1: p_treatment ≠ p_control

**Significance level (α):** 0.05

**Decision rule:**
- If p-value < α → Reject H0 (statistically significant difference)
- If p-value ≥ α → Fail to reject H0 (no statistically significant difference)

In [66]:
import pandas as pd
from scipy.stats import chi2_contingency

# STEP 1: Crosstab — just shows raw observed counts (NOT a conclusion by itself)
contingency = pd.crosstab(df['group'], df['converted'])
print("Observed counts:\n", contingency)

# STEP 2: Chi-square test — THIS is what actually tests significance
chi2, p_value, dof, expected = chi2_contingency(contingency)

print(f"\nChi2 statistic: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")

# STEP 3: Compare p-value to alpha — THIS gives the actual significance conclusion
alpha = 0.05
print("\n--- SIGNIFICANCE RESULT ---")
if p_value < alpha:
    print(f"p-value ({p_value:.4f}) < alpha ({alpha})")
    print("=> SIGNIFICANT difference exists between old_page and new_page conversion.")
else:
    print(f"p-value ({p_value:.4f}) >= alpha ({alpha})")
    print("=> NO significant difference between old_page and new_page conversion.")

Observed counts:
 converted       0      1
group                   
control    127785  17489
treatment  128046  17264

Chi2 statistic: 1.7036
P-value: 0.1918

--- SIGNIFICANCE RESULT ---
p-value (0.1918) >= alpha (0.05)
=> NO significant difference between old_page and new_page conversion.
